# 06 · Gradient descent, momentum and Adam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/math/06_gradient_descent.ipynb) &nbsp; 📖 Concept deck: [Optimization basics](https://nilesh-salpe.github.io/visual-ai/math/09-optimization-basics/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** gradient descent on an ill-conditioned quadratic, the 2/λ_max stability limit, momentum, and Adam from its update equations.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def make_quadratic(kappa):
    """f(x) = 0.5 * x^T diag(1, kappa) x  -> curvatures 1 and kappa."""
    H = np.diag([1.0, kappa])
    return (lambda x: 0.5 * x @ H @ x), (lambda x: H @ x)
X0 = np.array([1.0, 1.0])

## Task 1: Plain gradient descent

Run $x \leftarrow x - \eta \nabla f(x)$ for `steps` steps from `x0` and return the array of losses (length steps + 1, including the start).
On a quadratic, GD converges only if $\eta < 2/\lambda_{\max}$; the slowest direction then shrinks by $|1 - \eta\lambda_{\min}|$ per step.

In [ ]:
def gd(f, grad, x0, lr, steps):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_gd(f):
    fn, g = make_quadratic(100)
    ok = f(fn, g, X0, 0.019, 300); bad = f(fn, g, X0, 0.021, 300)
    assert len(ok) == 301
    assert ok[-1] < 1e-2 * ok[0], "lr below 2/λmax = 0.02 must converge"
    assert bad[-1] > bad[0], "lr above 2/λmax = 0.02 must diverge"

check("gd", _test_gd, gd)

## Task 2: Heavy-ball momentum

Keep a velocity: $v \leftarrow \beta v + \nabla f(x)$, $x \leftarrow x - \eta v$. Return losses as before.
Momentum averages away the zig-zag across the steep direction and accelerates along the shallow one.

In [ ]:
def momentum(f, grad, x0, lr, beta, steps):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_momentum(f):
    fn, g = make_quadratic(100)
    plain = gd_solution(fn, g, X0, 0.019, 200) if 'gd_solution' in globals() else None
    m = f(fn, g, X0, 0.019, 0.8, 200)
    assert len(m) == 201
    assert m[-1] < 1e-6, f"momentum should reach < 1e-6 in 200 steps, got {m[-1]:.2e}"
    assert plain is None or m[-1] < plain[-1], "momentum should beat plain GD here"

check("momentum", _test_momentum, momentum)

## Task 3: Adam

Kingma & Ba (2014): $m \leftarrow \beta_1 m + (1-\beta_1) g$, $v \leftarrow \beta_2 v + (1-\beta_2) g^2$, bias-correct $\hat m = m/(1-\beta_1^t)$, $\hat v = v/(1-\beta_2^t)$, then $x \leftarrow x - \eta\, \hat m / (\sqrt{\hat v} + \epsilon)$.
Defaults from the paper: β₁ = 0.9, β₂ = 0.999, ε = 10⁻⁸. Return the losses.

In [ ]:
def adam(f, grad, x0, lr, steps, b1=0.9, b2=0.999, eps=1e-8):
    # TODO (t starts at 1)
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_adam(f):
    fn, g = make_quadratic(100)
    a = f(fn, g, X0, 0.05, 400)
    assert len(a) == 401 and a[-1] < 1e-3 * a[0], f"Adam should converge, got {a[-1]:.2e}"
    fn2, g2 = make_quadratic(1e4)
    a2 = f(fn2, g2, X0, 0.05, 400)
    assert a2[-1] < 1e-3 * a2[0], "Adam's per-coordinate scaling makes it robust to κ = 10^4 at the same lr"

check("adam", _test_adam, adam)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Plain gradient descent
def gd_solution(f, grad, x0, lr, steps):
    x = x0.astype(float).copy(); out = [f(x)]
    for _ in range(steps):
        x = x - lr * grad(x); out.append(f(x))
    return np.array(out)

check("gd (reference)", _test_gd, gd_solution)

In [ ]:
# @title Solution for Task 2: Heavy-ball momentum
def momentum_solution(f, grad, x0, lr, beta, steps):
    x = x0.astype(float).copy(); v = np.zeros_like(x); out = [f(x)]
    for _ in range(steps):
        v = beta * v + grad(x); x = x - lr * v; out.append(f(x))
    return np.array(out)

check("momentum (reference)", _test_momentum, momentum_solution)

In [ ]:
# @title Solution for Task 3: Adam
def adam_solution(f, grad, x0, lr, steps, b1=0.9, b2=0.999, eps=1e-8):
    x = x0.astype(float).copy(); m = np.zeros_like(x); v = np.zeros_like(x); out = [f(x)]
    for t in range(1, steps + 1):
        gr = grad(x); m = b1 * m + (1 - b1) * gr; v = b2 * v + (1 - b2) * gr * gr
        x = x - lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps); out.append(f(x))
    return np.array(out)

check("adam (reference)", _test_adam, adam_solution)

---
## Experiments
**Condition number vs steps.** With the best fixed learning rate, plain GD needs roughly κ·ln(1/ε)/2 steps; momentum needs roughly √κ times fewer. Count steps to reach 10⁻⁶ of the start.

In [ ]:
for kappa in [10, 100, 1000]:
    fn, g = make_quadratic(kappa); lr = 2 / (1 + kappa)
    def steps_to(losses): return int(np.argmax(losses < 1e-6 * losses[0])) if (losses < 1e-6 * losses[0]).any() else None
    b = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)) ** 2; lr_m = (2 / (1 + np.sqrt(kappa))) ** 2
    print(f"kappa={kappa:5d}: GD {steps_to(gd_solution(fn, g, X0, lr, 20000))} steps, momentum {steps_to(momentum_solution(fn, g, X0, lr_m, b, 20000))} steps")

---
## Takeaways
* GD is stable only for η < 2/λmax, and the smallest curvature then sets the speed: ill-conditioning (large κ) makes it slow.
* Momentum cuts the dependence on κ to about √κ.
* Adam rescales each coordinate by its gradient RMS, making it far less sensitive to curvature differences.